# py-condiments — function-by-function R⇄Py parity

## 1. Setup

In [1]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
NB = Path('.').resolve(); PORT = NB.parent if NB.name=='examples' else NB
sys.path.insert(0, str(PORT))
sys.path.insert(0, str(PORT.parent/'omicverse-rebuildr'/'engine'))
import pycondiments as cd
from parity_metrics import compute_parity
ref = json.loads((PORT/'data'/'reference_output.json').read_text())
cand = json.loads((PORT/'data'/'candidate_output.json').read_text())

## 2. Function-by-function

### 2.1 `imbalance_score`

| R | Python | Default | Description |
|---|---|---|---|
| `rd` | `rd` | — | reduced-dim coords (n × d) |
| `conditions` | `conditions` | — | per-cell labels |
| `k` | `k` | `10` | kNN size |
| `smooth` | `smooth` | `k` | GAM (R) / kNN-average (Py) smoothing |

```r
imb <- condiments::imbalance_score(rd, conditions, k = 10, smooth = 10)
```

In [2]:
# Comparison: imbalance_score Pearson vs R
r_imb = np.array(ref['imbalance']['score']); p_imb = np.array(cand['imbalance']['score'])
mask = np.isfinite(r_imb) & np.isfinite(p_imb)
m = compute_parity(r_imb[mask], p_imb[mask], 'ordinal')
print(f'imbalance score Pearson: {m:.4f}  →  {"✅" if m > 0.95 else "🟡"}')

imbalance score Pearson: 1.0000  →  ✅


### 2.2 `progressionTest`

| R | Python | Default | Description |
|---|---|---|---|
| `sds` | `pseudotime` (or `sds={pseudotime, cellWeights}`) | — | Slingshot output |
| `conditions` | `conditions` | — | per-cell labels |
| `global` | `global_test` | TRUE/True | combine across lineages |

```r
ptest <- condiments::progressionTest(sds, conditions)
```

In [3]:
print(f"R p-value:  {ref['progression']['pvalue']:.4g}")
print(f"Py p-value: {cand['progression']['pvalue']:.4g}")

R p-value:  1.301e-05
Py p-value: 3.375e-06


### 2.3 `topologyTest`

R: re-fits Slingshot per condition + compares trajectories.
Py: simplified χ² on dominant-lineage contingency (v0.1 limitation; see RECONSTRUCTION_REPORT.md §6).

In [4]:
print(f"R p-value:  {ref['topology']['pvalue']:.4g}")
print(f"Py p-value: {cand['topology']['pvalue']:.4g}")

R p-value:  2.2e-16
Py p-value: 1


### 2.4 `differentiationTest`

| R | Python | Default | Description |
|---|---|---|---|
| `sds` | `cellWeights` | — | lineage weights matrix |
| `conditions` | `conditions` | — | per-cell labels |

Requires ≥ 2 lineages.

### 2.5 `weights_from_pst`, `merge_sds`

Utility functions; trivial — see [tutorial_toy.ipynb](tutorial_toy.ipynb) §4.5.

## 3. Aggregate verdict

| Function | Output | Metric | Pass |
|---|---|---|---|
| `imbalance_score` | per-cell score | Pearson 1.000 | ✅ |
| `progressionTest` | global p-value | both <0.05 → agree | ✅ |
| `topologyTest` | global p-value | simplified (v0.1) | 🟡 |
| `differentiationTest` | global p-value | (requires multi-lineage) | (untested in single-lineage fixture) |